# Water Indices: NDWI and MNDWI

This notebook demonstrates the **Normalized Difference Water Index (NDWI)** and
**Modified Normalized Difference Water Index (MNDWI)** functions in xarray-spatial.

- **NDWI** (McFeeters 1996): `(Green - NIR) / (Green + NIR)` -- highlights open water while suppressing vegetation and soil.
- **MNDWI** (Xu 2006): `(Green - SWIR) / (Green + SWIR)` -- works better in urban areas by reducing false positives from built-up surfaces.

Both return values in [-1, 1]. Positive values generally indicate water.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from xrspatial.multispectral import ndwi, mndwi

## Create synthetic band data

We build a simple scene with a water body (high green reflectance, low NIR/SWIR)
surrounded by vegetation (low green, high NIR) and bare soil.

In [ ]:
np.random.seed(42)
rows, cols = 100, 100
y = np.arange(rows)
x = np.arange(cols)

# Create a circular water body in the center
yy, xx = np.meshgrid(y, x, indexing='ij')
dist = np.sqrt((yy - 50)**2 + (xx - 50)**2)
water_mask = dist < 20
veg_mask = (dist >= 20) & (dist < 40)
soil_mask = dist >= 40

# Green band: water=0.3, vegetation=0.05, soil=0.15
green = np.where(water_mask, 0.30, np.where(veg_mask, 0.05, 0.15))
green += np.random.normal(0, 0.01, green.shape)

# NIR band: water=0.02, vegetation=0.45, soil=0.25
nir = np.where(water_mask, 0.02, np.where(veg_mask, 0.45, 0.25))
nir += np.random.normal(0, 0.01, nir.shape)

# SWIR band: water=0.01, vegetation=0.20, soil=0.35
swir = np.where(water_mask, 0.01, np.where(veg_mask, 0.20, 0.35))
swir += np.random.normal(0, 0.01, swir.shape)

green_da = xr.DataArray(green, dims=['y', 'x'], coords={'y': y, 'x': x})
nir_da = xr.DataArray(nir, dims=['y', 'x'], coords={'y': y, 'x': x})
swir_da = xr.DataArray(swir, dims=['y', 'x'], coords={'y': y, 'x': x})

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
green_da.plot(ax=axes[0], cmap='Greens')
axes[0].set_title('Green Band')
nir_da.plot(ax=axes[1], cmap='Reds')
axes[1].set_title('NIR Band')
swir_da.plot(ax=axes[2], cmap='copper')
axes[2].set_title('SWIR Band')
plt.tight_layout()
plt.show()

## Compute NDWI

`ndwi(green, nir)` returns `(Green - NIR) / (Green + NIR)`. Water pixels produce
positive values because green reflectance exceeds NIR over water.

In [ ]:
ndwi_result = ndwi(green_da, nir_da)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ndwi_result.plot(ax=axes[0], cmap='RdYlBu', vmin=-1, vmax=1)
axes[0].set_title('NDWI')

# Threshold at 0 to create a binary water mask
water_detected = (ndwi_result > 0).astype(float)
water_detected.plot(ax=axes[1], cmap='Blues', vmin=0, vmax=1)
axes[1].set_title('NDWI > 0 (water mask)')
plt.tight_layout()
plt.show()

## Compute MNDWI

`mndwi(green, swir)` substitutes SWIR for NIR. This reduces false water
detections in built-up areas where NIR can be ambiguous.

In [ ]:
mndwi_result = mndwi(green_da, swir_da)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
mndwi_result.plot(ax=axes[0], cmap='RdYlBu', vmin=-1, vmax=1)
axes[0].set_title('MNDWI')

water_detected_m = (mndwi_result > 0).astype(float)
water_detected_m.plot(ax=axes[1], cmap='Blues', vmin=0, vmax=1)
axes[1].set_title('MNDWI > 0 (water mask)')
plt.tight_layout()
plt.show()

## Compare NDWI vs MNDWI

Side-by-side comparison shows that both indices detect the same water body.
The difference between them is most visible in urban or mixed-use scenes
where built-up surfaces can confuse NDWI.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

ndwi_result.plot(ax=axes[0], cmap='RdYlBu', vmin=-1, vmax=1)
axes[0].set_title('NDWI (Green vs NIR)')

mndwi_result.plot(ax=axes[1], cmap='RdYlBu', vmin=-1, vmax=1)
axes[1].set_title('MNDWI (Green vs SWIR)')

diff = mndwi_result - ndwi_result
diff.plot(ax=axes[2], cmap='coolwarm', center=0)
axes[2].set_title('MNDWI - NDWI')
plt.tight_layout()
plt.show()

## Using the accessor interface

You can also call NDWI and MNDWI through the `.xrs` accessor on any DataArray.
When called on the green band, pass the other band as the first argument.

In [ ]:
import xrspatial  # registers .xrs accessor

# Accessor: self = green band
ndwi_acc = green_da.xrs.ndwi(nir_da)
mndwi_acc = green_da.xrs.mndwi(swir_da)

# Verify results match
assert np.allclose(ndwi_result.values, ndwi_acc.values, equal_nan=True)
assert np.allclose(mndwi_result.values, mndwi_acc.values, equal_nan=True)
print('Accessor results match direct function calls.')